# AD xQTL loci table

Reproducible pipeline to generate the unified AD xQTL loci summary Excel table and deploy the Shiny explorer.

You will need 5 Metadata:
- `metadata_analysis.csv` containing all exported tables paths
- `contexts_metadata.csv` containing contexts (datasets) label mapping between the different analysis/exported tables
- `columns_metadata.tsv` containing all columns to keep for the excel sheets, with associated metadata (column width, coloring..)
- `excel_metadata.tsv` containing some meta information for the excel sheet construction
- `pattern_coloring.tsv` containing all pattern to color in the excel sheet
  
1 metadata is optional - `long_table_columns_selection.csv` to generate a long table with selected columns from the table. 

`xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz` generated in step III 
(in this table each row is a variant-ADlocus-Method-context-gene_name information, and so facilitate querying informations ) 



## How to Add New Data

To add a new study, cohort, or GWAS, you need to:
1. Upload files to S3 and download to cluster
2. Update the two metadata files
3. Rerun the R pipeline on BU SCC to regenerate the summary `.csv.gz` files
4. Rerun the code cell below — no changes to the code needed

---

### 1. Upload to S3 and download to cluster

```bash
# Upload
aws s3 cp new_study.exported.toploci.bed.gz \
  s3://statfungen/ftp_fgc_xqtl/analysis_result/single_context/NEW_STUDY/export/summary/context_specific/

# Download to cluster
aws s3 sync s3://statfungen/ftp_fgc_xqtl/analysis_result/single_context/NEW_STUDY/ \
  /data/analysis_result/single_context/NEW_STUDY/
```

---

### 2. Update `metadata_analysis.csv`

This file controls which input files are read for each method. Add one row per new method.
To see its current structure:
```r
mtd <- fread(file.path(out, 'metadata_analysis.csv'))
head(mtd)
```
To append a new row:
```r
mtd <- fread(file.path(out, 'metadata_analysis.csv'))
mtd <- rbind(mtd, data.table(
  `Data Type` = 'eQTL',
  Cohort      = 'NEW_COHORT',
  Modality    = 'snRNA-seq',
  context     = 'NEW_context',
  Method      = 'single_context_finemapping',   # or ColocBoost, AD_GWAS_finemapping, etc.
  Path        = 'analysis_result/single_context/NEW_STUDY/export/summary/context_specific/NEW_file.bed.gz'
), fill = TRUE)
fwrite(mtd, file.path(out, 'metadata_analysis.csv'))
```

---

### 3. Update `contexts_metadata.csv`

This file maps context IDs to their display labels and determines which Excel sheet they appear in.
To see its current structure:
```r
contexts <- fread(file.path(out, 'contexts_metadata.csv'))
head(contexts)
```
To append a new context:
```r
contexts <- fread(file.path(out, 'contexts_metadata.csv'))
contexts <- rbind(contexts, data.table(
  context        = 'NEW_context',
  context_broad  = 'snuc_NEW_eQTL',
  context_short  = 'NEW_eQTL',
  context_hi     = 'NEW_eQTL',
  context_coloc  = 'NEW_context',
  context_broad2 = 'Exc',     # which Excel sheet: Brain / Exc / Inh / Oli / OPC / Ast / Mic / Immune (bulk)
  qtl_type       = 'eQTL'
), fill = TRUE)
fwrite(contexts, file.path(out, 'contexts_metadata.csv'))
```
> ⚠️ If `context_broad2` is a completely new category not already in the Excel,
> also add it to `columns_metadata.tsv` and `excel_metadata.tsv` to create a new sheet.

---

### 4. Rerun R pipeline

From `main_text/5_AD_xQTL_genes_cis_trans/staging/gene_priorization_table/`, run in this exact order:
```bash
Rscript gene_locus_summary.R   # Sections I–III
Rscript gwas_cs_extension.R    # between Sections I and II
Rscript export_mr.R            # before MR integration in Section III
```
This regenerates:
```r
# These two files are what the code cell below reads
res_adx  <- fread(file.path(out, 'xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz'))
res_adfv <- fread(file.path(out, 'AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz'))
```

---

### 5. Rerun the code cell below

No changes needed. The loop below automatically picks up any new `context_broad2` value
that appears in `excel_metadata.tsv`:
```r
for(cont in colsmtd[wildcard=='context_broad2']$r_name){ ... }
```


## Libraries

In [6]:
library(data.table)
library(openxlsx)
library(stringr)

## Set up working directory

In [7]:
out <- '/restricted/projectnb/xqtl/jaempawi/xqtl/AD_xQTL_loci/'
source(file.path(out, 'gene_prio_utils.R'))

1 threads available for data.table



In [8]:
options(warn = -1)

# Load inputs
res_adx   <- fread(file.path(out, 'xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz'))
res_adfv  <- fread(file.path(out, 'AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz'))

# ── Add trans-eQTL (DeJager 6ct) + ColocBoost (ROSMAP) ──────────────────────
gene_ref_file <- file.path(out, 'Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list')
gene_map      <- fread(gene_ref_file, col.names=c('chr','start','end','gene_ID','gene_name'))
gene_id2name  <- gene_map[, setNames(gene_name, gene_ID)]
ad_vars       <- unique(res_adfv[, .(variant_ID, ADlocus, ADlocusID, locus_index)])

# 1. Trans-eQTL -------------------------------------------------------
trans_files <- c(
  Ast=file.path(out,'Ast_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Exc=file.path(out,'Exc_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Inh=file.path(out,'Inh_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Mic=file.path(out,'Mic_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Oli=file.path(out,'Oli_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  OPC=file.path(out,'OPC_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz')
)
trans_files <- trans_files[file.exists(trans_files)]
if (length(trans_files) > 0) {
  ctx_b2 <- c(Ast='Ast',Exc='Exc',Inh='Inh',Mic='Mic',Oli='Oli',OPC='OPC')
  trans_rows <- rbindlist(lapply(names(trans_files), function(ct) {
    dt <- fread(trans_files[[ct]])
    if ('#chr' %in% names(dt)) setnames(dt,'#chr','chr')
    dt[, gene_ID        := str_extract(event_ID,'ENSG[0-9]+$')]
    dt[, gene_name      := gene_id2name[gene_ID]]
    dt[, context        := paste0(ct,'_DeJager_eQTL')]
    dt[, context_short  := context]
    dt[, context_broad2 := ctx_b2[[ct]]]
    dt[, qtl_type       := 'eQTL']
    dt[, Method         := 'trans_finemapping']
    merge(dt, ad_vars, by='variant_ID', all=FALSE)[,
      .(ADlocus,ADlocusID,locus_index,variant_ID,gene_ID,gene_name,
        context,context_short,context_broad2,qtl_type,Method,PIP,z,purity,event_ID,region_ID)]
  }), fill=TRUE)
  message('Trans: ', nrow(trans_rows), ' rows | ',
          uniqueN(trans_rows$variant_ID), ' variants | ',
          uniqueN(trans_rows$gene_ID), ' genes')
  print(trans_rows[, .N, by=context])
  res_adx <- rbind(res_adx, trans_rows, fill=TRUE)
} else { message('No trans files found in out') }

# 2. ColocBoost -------------------------------------------------------
cb_file <- file.path(out,'AD_xQTL_ROSMAP_colocboost_export_release.bed')
if (file.exists(cb_file)) {
  cb <- fread(cb_file)
  if ('#chr' %in% names(cb)) setnames(cb,'#chr','chr')
  cb[, gene_ID   := region_ID]
  cb[, gene_name := gene_id2name[gene_ID]]
  cb[, coef_mean := sapply(str_split(coef,';'), function(x) mean(as.numeric(x),na.rm=TRUE))]
  cb[, context        := 'ColocBoost_ROSMAP']
  cb[, context_short  := 'ColocBoost_ROSMAP']
  cb[, context_broad2 := 'Brain']
  cb[, qtl_type       := 'eQTL']
  cb[, Method         := 'AD_xQTL_colocalization']
  cb_rows <- merge(cb, ad_vars, by='variant_ID', all=FALSE)[,
    .(ADlocus,ADlocusID,locus_index,variant_ID,gene_ID,gene_name,
      context,context_short,context_broad2,qtl_type,Method,
      vcp,coef=coef_mean,event_ID,cos_ID,cos_npc)]
  message('ColocBoost: ', nrow(cb_rows), ' rows | ',
          uniqueN(cb_rows$variant_ID), ' variants | ',
          uniqueN(cb_rows$gene_ID), ' genes')
  res_adx <- rbind(res_adx, cb_rows, fill=TRUE)
} else { message('ColocBoost file not found') }

rm(gene_map, gene_id2name, ad_vars)
message('Total res_adx after injection: ', nrow(res_adx), ' rows')


# Load metadata
cols      <- fread(file.path(out, 'columns_metadata.tsv'))[(keep==1)]
colsmtd   <- fread(file.path(out, 'excel_metadata.tsv'))
colorsmtd <- fread(file.path(out, 'pattern_coloring.tsv'))

# Summarise first so n_trans_genes / trans_genes are populated
res_adx <- suppressWarnings(SummarizeTable(res_adx, group.by='context_short'))

# Wide table
res_adxub <- WideTable(res_adx, split.by=c('context_broad2','qtl_type'))

# Filter top variants
res_adxub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
            ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
               (rank(pval)==1&!is.na(pval)))), by='locus_index']
res_adxubf <- res_adxub[(top_variants)][!is.na(locus_index)][variant_ID!='']

# Prep columns metadata
cols <- PrepColsMtd(cols, colsmtd, res_adxubf)

# Main sheet
wb <- CreateExcelFormat(res_adxubf, columns_mtd=cols, colors=colorsmtd)

# One sheet per broad context
for(cont in colsmtd[wildcard=='context_broad2']$r_name){
  message(cont)
  res_adxc   <- res_adx[context_broad2==cont]
  res_adxc   <- SummarizeTable(res_adxc, group.by='qtl_type')
  res_adxcub <- WideTable(res_adxc)
  
  res_adxcub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
               ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
                  (rank(pval)==1&!is.na(pval)))), by='ADlocus']
  res_adxcubf <- res_adxcub[(top_variants)]
  
  res_adxcubf[, gwas_significance:=ifelse(min_pval<5e-8,'genome wide',
                                    ifelse(min_pval<1e-6,'suggestive','ns'))]
  res_adxcubf[, mlog10pval:=-log10(min_pval)]
  res_adxcubf[, top_confidence:=str_extract(xQTL_effects,'CL[0-9]')]
  
  wb <- CreateExcelFormat(res_adxcubf, columns_mtd=cols, colors=colorsmtd,
                          wb=wb, sheet_name=cont)
}

# Save
saveWorkbook(wb, file.path(out, 'unified_AD_loci_xQTL_summary.xlsx'), overwrite=TRUE)

Trans: 192644 rows | 4520 variants | 2084 genes



            context      N
             <char>  <int>
1: Ast_DeJager_eQTL   3095
2: Exc_DeJager_eQTL 141655
3: Inh_DeJager_eQTL  36420
4: Mic_DeJager_eQTL   5044
5: Oli_DeJager_eQTL   1267
6: OPC_DeJager_eQTL   5163


ColocBoost: 26220 rows | 5144 variants | 151 genes

Total res_adx after injection: 2576237 rows

summarizing per context_short the xQTL evidence for each variant

generating columns per context_broad2 and qtl_type

Brain eQTL

Inh eQTL

Mic eQTL

OPC eQTL

Oli eQTL

Brain mQTL

Exc eQTL

Immune (bulk) eQTL

Brain pQTL

Brain sQTL

Ast eQTL

Exc sQTL

Brain p_sQTL

Brain u_sQTL

Inh sQTL

Oli sQTL

Ast sQTL

Brain gpQTL

Brain haQTL

OPC sQTL

Mic sQTL

Ast caQTL

Mic caQTL

pvalue_sexFbeta_sexFse_sexFpvalue_sexF_interactionbeta_sexF_interactionse_sexF_interactionpvalue_sexbeta_sexse_sexpvalue_sex_interactionbeta_sex_interactionse_sex_interactionpvalue_genderbeta_genderse_genderpvalue_gender_interactionbeta_gender_interactionse_gender_interactionresourceMAFbetahatsebetahatpnum_CSnum_IVcpipmeta_effse_meta_effmeta_pvalQQ_pvalI2gwas_studymolecular_idmethodis_imputableis_selected_methodrsq_cvpval_cvtwas_ztwas_pvaltypeblockidregion_idsusie_pipmucsfiletrans_genestrans_contextspuritymax_APOE4_

## Generate Shiny data.csv

In [9]:
# ── Extract shiny_app/data.csv from the generated Excel ──────────────────────
library(readxl)
shiny_app_dir <- file.path(out, 'shiny_app')
xl_path       <- file.path(out, 'unified_AD_loci_xQTL_summary.xlsx')

raw_hdrs  <- suppressMessages(read_excel(xl_path, sheet=1, col_names=FALSE, n_max=3))
super_hdr <- zoo::na.locf(as.character(raw_hdrs[1,]), na.rm=FALSE)
col_names <- as.character(raw_hdrs[3,])
dat <- suppressMessages(read_excel(xl_path, sheet=1, skip=3,
                                   col_names=col_names, .name_repair='minimal'))

ci     <- function(nm) which(col_names==nm)[1]
ci_all <- function(nm) which(col_names==nm)

ct_map <- c('Brain xQTL'='ct_Brain_xQTL','Exc xQTL'='ct_Exc_xQTL',
            'Inh xQTL'='ct_Inh_xQTL','Oli xQTL'='ct_Oli_xQTL',
            'OPC xQTL'='ct_OPC_xQTL','Ast xQTL'='ct_Ast_xQTL',
            'Microglia xQTL'='ct_Microglia_xQTL','Bulk Immune xQTL'='ct_Bulk_Immune_xQTL')
ct_block <- lapply(names(ct_map), function(s) which(super_hdr==s))

shiny_dat <- data.frame(
  locus_index=dat[[ci('locus index')]],ADlocus=dat[[ci('ADlocus')]],
  ADlocusID=dat[[ci('ADlocusID')]],n_variants=dat[[ci('# variants in this locus')]],
  n_variants_01=dat[[ci('# with inclusion score >01')]],
  variant_ID=dat[[ci('Variant ID')]],rsid=dat[[ci('Rsid')]],
  chr=dat[[ci('Chr')]],pos=dat[[ci('Pos')]],effect_allele=dat[[ci('Effect allele')]],
  cv2f_score=dat[[ci('cV2F score')]],max_inclusion=dat[[ci('maximum inclusion score')]],
  max_inclusion_method=dat[[ci('maximum inclusion score method')]],
  cv2f_rank=dat[[ci('cV2F rank')]],max_zscore=dat[[ci('Maximum zscore')]],
  min_pval=dat[[ci('Min p-value')]],significance=dat[[ci('Significance')]],
  log10pval=dat[[ci('log10(pval)')]],
  gwas_assoc=dat[[ci('GWAS associated to this variant')]],
  gwas_src1=dat[[ci_all('GWAS Sources')[1]]],gwas_src2=dat[[ci_all('GWAS Sources')[2]]],
  gwas_src3=dat[[ci_all('GWAS Sources')[3]]],gwas_src4=dat[[ci_all('GWAS Sources')[4]]],
  detected_methods=dat[[ci('Detected methods')]],
  cs_coverage=dat[[ci('Credible set coverage (SuSiE)')]],
  gene=dat[[ci('xQTL target gene')]],gene_id=dat[[ci('gene ID')]],
  tss=dat[[ci('tss')]],tes=dat[[ci('tes')]],
  dist_tss=dat[[ci('Distance from TSS')]],dist_tes=dat[[ci('Distance from TES')]],
  max_twas_z=dat[[ci('Maximum TWAS zscore')]],max_twas_ctx=dat[[ci('Maximum TWAS context')]],
  twas_sig=dat[[ci('TWAS significant')]],mr_sig=dat[[ci('MR significant')]],
  ctwas_sig=dat[[ci('cTWAS significant')]],
  xqtl_max_inclusion=dat[[ci('Maximum inclusion score')]],
  variant_rank=dat[[ci('Variant rank')]],context=dat[[ci('Context')]],
  effect=dat[[ci('effect')]],n_variants_loci=dat[[ci('# variants within the loci')]],
  n_contexts=dat[[ci('# molecular contexts (datasets)')]],
  top_confidence=dat[[ci('max confidence level')]]
)
for (k in seq_along(ct_map)) {
  idxs <- ct_block[[k]]
  shiny_dat[[ct_map[k]]] <- if (length(idxs))
    apply(dat[,idxs,drop=FALSE],1,function(r) any(!is.na(r))) else FALSE
}
ctx_idxs <- ci_all('Ordered contexts (confidence level , n datasets)')
shiny_dat$ordered_contexts <- apply(dat[,ctx_idxs,drop=FALSE],1,function(r){
  v <- r[!is.na(r) & nchar(trimws(as.character(r)))>0]
  if(length(v)) paste(v,collapse=' | ') else NA_character_
})
shiny_dat$n_trans_genes    <- suppressWarnings(as.integer(dat[[ci('# trans genes')]]))
shiny_dat$n_trans_contexts <- suppressWarnings(as.integer(dat[[ci('# trans contexts')]]))
# trans_genes / trans_contexts: build from bed.gz files (Excel only has counts)
tg_i <- ci('trans genes');    shiny_dat$trans_genes    <- if(!is.na(tg_i)) dat[[tg_i]] else NA_character_
tc_i <- ci('trans contexts'); shiny_dat$trans_contexts <- if(!is.na(tc_i)) dat[[tc_i]] else NA_character_
shiny_dat$n_trans_genes[!is.na(shiny_dat$n_trans_genes)&shiny_dat$n_trans_genes==0]          <- NA_integer_
shiny_dat$n_trans_contexts[!is.na(shiny_dat$n_trans_contexts)&shiny_dat$n_trans_contexts==0] <- NA_integer_

# Populate trans_genes and trans_contexts from the bed.gz files
trans_bed_files <- c(
  Ast=file.path(out,'Ast_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Exc=file.path(out,'Exc_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Inh=file.path(out,'Inh_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Mic=file.path(out,'Mic_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  Oli=file.path(out,'Oli_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz'),
  OPC=file.path(out,'OPC_DeJager_eQTL.trans_single_gene.maf005.withoutHF.exported.toploci.bed.gz')
)
trans_bed_files <- trans_bed_files[file.exists(trans_bed_files)]
if (length(trans_bed_files) > 0) {
  # Gene ref — try same candidate paths as Cell 6
  gene_ref_candidates2 <- c(
    file.path(out, 'Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list'),
    '/data/resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list',
    '/restricted/projectnb/xqtl/xqtl_protocol/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list'
  )
  gene_ref_file2 <- gene_ref_candidates2[file.exists(gene_ref_candidates2)][1]
  if (is.na(gene_ref_file2)) {
    message('  WARNING: gene region list not found — trans_genes will be empty')
  } else {
    gmap2     <- fread(gene_ref_file2, col.names=c('chr','start','end','gene_ID','gene_name'))
    gid2name2 <- gmap2[, setNames(gene_name, gene_ID)]
    ad_vids   <- unique(shiny_dat$variant_ID[!is.na(shiny_dat$variant_ID)])
    trans_all <- rbindlist(lapply(names(trans_bed_files), function(ct) {
      dt <- fread(trans_bed_files[[ct]])
      if ('#chr' %in% names(dt)) setnames(dt, '#chr', 'chr')
      dt[, gene_name := gid2name2[str_extract(event_ID, 'ENSG[0-9]+$')]]
      dt[, context   := ct]
      dt[variant_ID %in% ad_vids & !is.na(gene_name),
         .(variant_ID, gene_name, context)]
    }), fill=TRUE)
    trans_summ <- trans_all[, .(
      trans_genes    = paste(sort(unique(gene_name)), collapse='|'),
      trans_contexts = paste(sort(unique(context)),   collapse='|')
    ), by=variant_ID]
    shiny_dt <- as.data.table(shiny_dat)
    shiny_dt[trans_summ, on='variant_ID',
             `:=`(trans_genes=i.trans_genes, trans_contexts=i.trans_contexts)]
    shiny_dat <- as.data.frame(shiny_dt)
    message('  trans_genes populated for ',
            sum(!is.na(shiny_dat$trans_genes)), ' rows')
    rm(gmap2, gid2name2, trans_all, trans_summ, shiny_dt)
  }
}

write.csv(shiny_dat, file.path(shiny_app_dir,'data.csv'), row.names=FALSE, na='')
message('data.csv: ',nrow(shiny_dat),' rows x ',ncol(shiny_dat),' cols')
message('  trans-hit rows: ',sum(!is.na(shiny_dat$n_trans_genes)))


  trans_genes populated for 1108 rows

data.csv: 4795 rows x 56 cols

  trans-hit rows: 1063



## Deploy Shiny app

In [10]:
shiny_app_dir <- file.path(out, 'shiny_app')

# One-time: set credentials (get token at shinyapps.io/admin/#/tokens)
# rsconnect::setAccountInfo(name='jenny-empawi', token='<TOKEN>', secret='<SECRET>')

rsconnect::deployApp(
  appDir  = shiny_app_dir,
  appName = "AD-xQTL-Explorer",
  account = "jenny-empawi"
)

── Preparing for deployment ────────────────────────────────────────────────────

✔ Re-deploying "AD-xQTL-Explorer" using "server: shinyapps.io / username: jenny-empawi"

ℹ Looking up content with id "17020283"...

✔ Found content <https://jenny-empawi.shinyapps.io/AD-xQTL-Explorer/>

ℹ Bundling 3 files: DEPLOY.md, app.R, and data.csv

ℹ Capturing R dependencies

✔ Found 62 dependencies

✔ Created bundle of size: 323,987b

ℹ Uploading bundle...

✔ Uploaded bundle with id 11897808

── Deploying to server ─────────────────────────────────────────────────────────



Waiting for task: 1682830754
  building: Building image: 14810292
  building: Installing system dependencies
  building: Fetching packages
  building: Installing packages
  building: Installing files
  building: Pushing image: 14810292
  deploying: Starting instances
  rollforward: Activating new instances
  terminating: Stopping old instances


── Deployment complete ─────────────────────────────────────────────────────────

✔ Successfully deployed to <https://jenny-empawi.shinyapps.io/AD-xQTL-Explorer/>

